# Day 30 — Global Health Data Exchange Data Day
### #30DayChartChallenge | April 2026

**How the world gained six years of life.** Horizontal waterfall chart decomposing the net change in global life expectancy from 1990 to 2021 into each leading cause's contribution. Each row is one cause; each bar starts where the row above it ended, so the chart reads top-to-bottom as a cumulative running total. Defeating diarrhoeal and lower respiratory infections added the most years; stroke, cancer, ischaemic heart disease, neonatal disorders and tuberculosis each added another half-year-plus. COVID-19 took 1.6 years back in a single shock. Net result: +6.0 years.

**Tool:** `ggplot2` · horizontal waterfall (cumulative bars) with gains, losses and a final net total
**Data:** [IHME Global Burden of Disease Study 2021 — Findings Booklet](https://www.healthdata.org/sites/default/files/2024-05/GBD_2021_Booklet_FINAL_2024.05.16.pdf) (May 2024), p. 13. Original publication: GBD 2021 Causes of Death Collaborators, *The Lancet* 403:2100, 2024.
**Author:** Sharfudeen Yasar Arafath


In [1]:
library(ggplot2)
library(showtext)
library(sysfonts)
library(scales)
library(ragg)
library(dplyr)

font_add_google("Outfit",           "outfit")
font_add_google("Roboto Condensed", "roboto_condensed")
font_add_google("JetBrains Mono",   "jetbrains")
showtext_auto()
showtext_opts(dpi = 300)

bg_col   <- "#0a0e17"
text_col <- "#e0e6f0"
subtext  <- "#9eb3c8"
grid_col <- "#1a2030"

gain_col   <- "#5fc9a6"   # gains  — teal
other_col  <- "#b794f4"   # the "Other smaller gains" rollup — lavender
covid_col  <- "#ffb84d"   # COVID-19 — orange
loss_col   <- "#ff6b6b"   # other losses — red
net_col    <- "#ff9f5a"   # final net — warm orange

raw <- read.csv("../../data/day_30/gbd2021_life_expectancy_contributions.csv",
                stringsAsFactors = FALSE, fileEncoding = "UTF-8")

# Cumulative positions for the waterfall.
df <- raw |>
  mutate(
    end       = cumsum(years),
    start     = lag(end, default = 0),
    direction = ifelse(years >= 0, "gain", "loss"),
    bar_col   = case_when(
      short_label == "COVID-19"                         ~ covid_col,
      short_label == "Other pandemic-related mortality" ~ loss_col,
      short_label == "Other smaller gains"              ~ other_col,
      TRUE                                              ~ gain_col
    )
  )

# Append a NET row that goes from 0 to the total — separated by a visual gap.
net_total <- sum(df$years)
net_row <- data.frame(
  cause       = "Net change, 1990 → 2021",
  short_label = "Net change",
  category    = "Net",
  years       = net_total,
  end         = net_total,
  start       = 0,
  direction   = "net",
  bar_col     = net_col
)

all_bars <- bind_rows(df, net_row)
# Row positions: top row = first contributor; spacer before NET row.
all_bars$row <- c(seq_len(nrow(df)), nrow(df) + 1.7)
# y is inverted so first row sits at the TOP.
y_max <- max(all_bars$row) + 0.6
all_bars$y <- y_max - all_bars$row

bar_h <- 0.62

# Connectors between consecutive contribution bars (skip before the NET row).
connectors <- df |>
  mutate(idx   = row_number(),
         row_a = idx,
         row_b = idx + 1,
         x     = end) |>
  filter(idx < nrow(df)) |>
  mutate(y_top = y_max - row_a + bar_h/2,
         y_bot = y_max - row_b - bar_h/2 + 1)
# correct y geometry: connector spans from end-of-bar-A (its x at row_a level)
# down to start-of-bar-B (same x at row_b level). y_a = bar A center, y_b = bar B center.
connectors <- connectors |>
  mutate(y_a = y_max - row_a,
         y_b = y_max - row_b)

# Pre-compute label x positions: gains label on the right of bar, losses on the left.
all_bars <- all_bars |>
  mutate(
    label_x   = ifelse(direction == "loss",
                       pmin(start, end) - 0.12,
                       pmax(start, end) + 0.12),
    label_h   = ifelse(direction == "loss", 1, 0),
    value_lbl = sprintf("%+.2f", years)
  )

# X-axis range with breathing room.
x_lim <- c(-0.6, 9.4)

p <- ggplot() +
  # vertical grid every 1 yr, light
  geom_vline(xintercept = seq(0, 9, 1),
             colour = grid_col, linewidth = 0.25) +
  # zero baseline
  geom_vline(xintercept = 0, colour = subtext, linewidth = 0.4) +

  # connectors between bars (vertical dotted lines)
  geom_segment(data = connectors,
               aes(x = x, xend = x,
                   y = y_max - row_a - bar_h/2,
                   yend = y_max - row_b + bar_h/2),
               colour = subtext, linewidth = 0.3, linetype = "13") +

  # bars (gains, losses, and NET)
  geom_rect(data = all_bars,
            aes(xmin = pmin(start, end),
                xmax = pmax(start, end),
                ymin = y - bar_h/2,
                ymax = y + bar_h/2,
                fill = I(bar_col))) +

  # value label at the outside tip of each bar
  geom_text(data = all_bars,
            aes(x = label_x, y = y, label = value_lbl,
                hjust = label_h, colour = I(bar_col)),
            family = "jetbrains", fontface = "bold", size = 4.0) +

  # cause label on the y-axis (left side, in its own gutter)
  geom_text(data = all_bars,
            aes(x = -3.7, y = y, label = short_label,
                colour = I(bar_col)),
            hjust = 0, family = "outfit", size = 3.8) +

  # Hero number for the NET total — big orange callout
  annotate("text",
           x = net_total + 1.5, y = (y_max - (nrow(df) + 1.7)),
           label = sprintf("+%.1f years", net_total),
           hjust = 0, vjust = 0.5,
           family = "outfit", fontface = "bold",
           size = 8.5, colour = net_col) +
  annotate("text",
           x = net_total + 1.5,
           y = (y_max - (nrow(df) + 1.7)) - 0.55,
           label = "of global life expectancy",
           hjust = 0, vjust = 0.5,
           family = "roboto_condensed",
           size = 3.6, colour = subtext) +

  # Faint section divider between GAINS rows and LOSSES rows
  annotate("segment",
           x = -3.8, xend = 9.2,
           y = y_max - which(df$direction == "loss")[1] + 0.5,
           yend = y_max - which(df$direction == "loss")[1] + 0.5,
           colour = grid_col, linewidth = 0.4, linetype = "32") +
  # And another faint divider between LOSSES and NET
  annotate("segment",
           x = -3.8, xend = 9.2,
           y = y_max - nrow(df) - 0.4,
           yend = y_max - nrow(df) - 0.4,
           colour = grid_col, linewidth = 0.4, linetype = "32") +

  scale_x_continuous(limits = c(-3.9, 9.7),
                     breaks = c(0, 2, 4, 6, 8),
                     labels = function(x) paste0("+", x, " yr"),
                     expand = c(0, 0),
                     position = "top") +
  scale_y_continuous(limits = c(-0.6, y_max + 0.9),
                     expand = c(0, 0)) +
  coord_cartesian(clip = "off") +
  labs(
    title    = "How the world gained six years of life",
    subtitle = paste0(
      "Years of global life expectancy gained or lost from each leading cause of death, 1990 → 2021. Bars stack cumulatively —\n",
      "each bar starts where the one above it ended. Enteric infections and lower respiratory infections added the most years; COVID-19 took 1.6 back."),
    caption  = paste(
      "Source: IHME Global Burden of Disease Study 2021 — Findings booklet, p. 13 (Years of life expectancy gained or lost from leading causes of death globally, 1990–2021).",
      "Original publication: GBD 2021 Causes of Death Collaborators, The Lancet, 2024.",
      "\"Other smaller gains\" aggregates 10 individually small contributors (other NCDs, COPD, unintentional injuries, measles, digestive, nutritional, transport injuries, suicide & homicide, malaria, diabetes & kidney diseases).",
      "#30DayChartChallenge · Day 30 · Global Health Data Exchange Data Day  |  Sharfudeen Yasar Arafath",
      sep = "\n"),
    x = NULL, y = NULL
  ) +
  theme_minimal(base_family = "outfit", base_size = 13) +
  theme(
    plot.background  = element_rect(fill = bg_col, colour = NA),
    panel.background = element_rect(fill = bg_col, colour = NA),
    panel.grid       = element_blank(),
    axis.text.y      = element_blank(),
    axis.text.x      = element_text(family = "jetbrains", size = 9,
                                    colour = subtext,
                                    margin = margin(b = 6)),
    axis.ticks       = element_blank(),
    plot.title       = element_text(family = "outfit", face = "bold",
                                    size = 30, colour = text_col,
                                    margin = margin(b = 6)),
    plot.subtitle    = element_text(family = "roboto_condensed",
                                    size = 13, colour = subtext,
                                    lineheight = 1.3,
                                    margin = margin(b = 22)),
    plot.caption     = element_text(family = "jetbrains", size = 8.4,
                                    colour = subtext, hjust = 0,
                                    lineheight = 1.5,
                                    margin = margin(t = 22)),
    plot.caption.position = "plot",
    plot.title.position   = "plot",
    plot.margin = margin(28, 32, 22, 32)
  )

agg_png("../../chart/day_30_ghdx.png",
        width = 13.5, height = 9.8, units = "in", res = 300,
        background = bg_col)
print(p)
invisible(dev.off())
cat("Saved chart/day_30_ghdx.png\n")


Warning message:
"package 'showtext' was built under R version 4.5.3"
Loading required package: sysfonts

Warning message:
"package 'sysfonts' was built under R version 4.5.3"
Loading required package: showtextdb

Warning message:
"package 'showtextdb' was built under R version 4.5.3"
Warning message:
"package 'ragg' was built under R version 4.5.3"

Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




Saved chart/day_30_ghdx.png
